In [1]:
import numpy as np
import pandas as pd
import string 
import matplotlib.pyplot as plt
import os 
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\Laptop
[nltk_data]     Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Laptop
[nltk_data]     Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
splits = {'train': 'train.csv', 'validation': 'validation.csv', 'test': 'test.csv'}
movies = pd.read_csv("hf://datasets/jquigl/imdb-genres/" + splits["train"])

C:\Users\Laptop Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print(movies.head(2))

      movie title - year    genre         expanded-genres  rating  \
0    Flaming Ears - 1992  Fantasy         Fantasy, Sci-Fi     6.0   
1  Jeg elsker dig - 1957  Romance  Comedy, Drama, Romance     5.8   

                                         description  
0  Flaming Ears is a pop sci-fi lesbian fantasy f...  
1  Six people - three couples - meet at random at...  


In [5]:
movies.shape

(238256, 5)

In [6]:
movies.iloc[0]

movie title - year                                  Flaming Ears - 1992
genre                                                           Fantasy
expanded-genres                                         Fantasy, Sci-Fi
rating                                                              6.0
description           Flaming Ears is a pop sci-fi lesbian fantasy f...
Name: 0, dtype: object

In [7]:
movies.columns

Index(['movie title - year', 'genre', 'expanded-genres', 'rating',
       'description'],
      dtype='object')

In [8]:
movies['genre'].value_counts()

genre
Thriller     36220
Romance      33704
Action       31156
Horror       24391
Crime        23368
Adventure    16234
Mystery      13159
Scifi        11363
Fantasy      11281
Family       10935
War           6415
History       5582
Biography     5438
Animation     5198
Sports        2997
Film-noir      815
Name: count, dtype: int64

In [9]:
#null value detection 
movies.isnull().sum()

movie title - year        0
genre                     0
expanded-genres           0
rating                69721
description               0
dtype: int64

In [10]:
#drop null value 
movies.dropna(inplace=True)

In [11]:
movies.shape

(168535, 5)

In [12]:
movies.duplicated().sum()

np.int64(2)

In [13]:
movies[movies.duplicated(keep=False)]

,movie title - year,genre,expanded-genres,rating,description
1089,Dead Man Running - 2009,Thriller,"Action, Crime, Drama",6.0,Nick's borrowed money from a loan shark. He's ...
3454,Spider's Web - 2002,Thriller,"Crime, Romance, Thriller",4.3,A wily businessman plots with a sultry executi...
125897,Spider's Web - 2002,Thriller,"Crime, Romance, Thriller",4.3,A wily businessman plots with a sultry executi...
145558,Dead Man Running - 2009,Thriller,"Action, Crime, Drama",6.0,Nick's borrowed money from a loan shark. He's ...


In [14]:
movies.duplicated().value_counts()

False    168533
True          2
Name: count, dtype: int64

In [15]:
movies = movies.drop_duplicates()

In [16]:
movies.shape

(168533, 5)

In [17]:
movies.duplicated().value_counts()

False    168533
Name: count, dtype: int64

In [18]:
print(type(movies['movie title - year'].iloc[0]))
print(type(movies['description'].iloc[0]))

<class 'str'>
<class 'str'>


In [19]:
movies.head()

,movie title - year,genre,expanded-genres,rating,description
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,Romance,"Comedy, Drama, Romance",5.8,Six people - three couples - meet at random at...
3,Gulliver Returns - 2021,Fantasy,"Animation, Adventure, Family",4.4,The legendary Gulliver returns to the Kingdom ...
5,Effie Gray - 2014,Romance,"Biography, Drama, Romance",6.0,A look at the scandalous love triangle between...
6,Indigo - 2008,Fantasy,"Action, Fantasy, Thriller",3.8,"A new generation of kids, called Indigo, who h..."


In [20]:
movies['expanded-genres']= movies['expanded-genres'].apply(lambda x:x.split())
movies['clean-description']= movies['description'].apply(lambda x:x.split())
movies.sample(4)

,movie title - year,genre,expanded-genres,rating,description,clean-description
22891,Adventures of Gustavas - 2014,Adventure,"[Animation,, Adventure,, Family]",6.1,When the insatiable discoverer Gustav catches ...,"[When, the, insatiable, discoverer, Gustav, ca..."
217867,El invierno en Lisboa - 1991,Romance,"[Drama,, Music,, Romance]",4.3,Jim is the drummer for the great black musicia...,"[Jim, is, the, drummer, for, the, great, black..."
102314,Hula - 1927,Romance,"[Drama,, Romance]",6.3,The daughter of a pineapple plantation owner i...,"[The, daughter, of, a, pineapple, plantation, ..."
134618,Leave It to the Irish - 1944,Action,"[Action,, Comedy,, Drama]",4.5,"Private Investigator Terry Moran, who is in lo...","[Private, Investigator, Terry, Moran,, who, is..."


In [21]:
movies.iloc[0]['clean-description']

['Flaming',
 'Ears',
 'is',
 'a',
 'pop',
 'sci-fi',
 'lesbian',
 'fantasy',
 'feature',
 'set',
 'in',
 'the',
 'year',
 '2700',
 'in',
 'the',
 'fictive',
 'burned-out',
 'city',
 'of',
 'Asche.',
 'It',
 'follows',
 'the',
 'tangled',
 'lives',
 'of',
 'three',
 'women',
 '-',
 'Volley,',
 'Nun',
 'and',
 'Spy.']

In [22]:
movies['tags']= movies['clean-description'] + movies['expanded-genres']

In [23]:
movies.head()

,movie title - year,genre,expanded-genres,rating,description,clean-description,tags
0,Flaming Ears - 1992,Fantasy,"[Fantasy,, Sci-Fi]",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...,"[Flaming, Ears, is, a, pop, sci-fi, lesbian, f...","[Flaming, Ears, is, a, pop, sci-fi, lesbian, f..."
1,Jeg elsker dig - 1957,Romance,"[Comedy,, Drama,, Romance]",5.8,Six people - three couples - meet at random at...,"[Six, people, -, three, couples, -, meet, at, ...","[Six, people, -, three, couples, -, meet, at, ..."
3,Gulliver Returns - 2021,Fantasy,"[Animation,, Adventure,, Family]",4.4,The legendary Gulliver returns to the Kingdom ...,"[The, legendary, Gulliver, returns, to, the, K...","[The, legendary, Gulliver, returns, to, the, K..."
5,Effie Gray - 2014,Romance,"[Biography,, Drama,, Romance]",6.0,A look at the scandalous love triangle between...,"[A, look, at, the, scandalous, love, triangle,...","[A, look, at, the, scandalous, love, triangle,..."
6,Indigo - 2008,Fantasy,"[Action,, Fantasy,, Thriller]",3.8,"A new generation of kids, called Indigo, who h...","[A, new, generation, of, kids,, called, Indigo...","[A, new, generation, of, kids,, called, Indigo..."


In [24]:
# select only desired columns 
new_df= movies[['movie title - year','rating', 'tags']]

In [25]:
new_df.head()

,movie title - year,rating,tags
0,Flaming Ears - 1992,6.0,"[Flaming, Ears, is, a, pop, sci-fi, lesbian, f..."
1,Jeg elsker dig - 1957,5.8,"[Six, people, -, three, couples, -, meet, at, ..."
3,Gulliver Returns - 2021,4.4,"[The, legendary, Gulliver, returns, to, the, K..."
5,Effie Gray - 2014,6.0,"[A, look, at, the, scandalous, love, triangle,..."
6,Indigo - 2008,3.8,"[A, new, generation, of, kids,, called, Indigo..."


In [26]:
# Converting list to str for tags 
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df.head()

C:\Users\Laptop Lenovo\AppData\Local\Temp\ipykernel_14064\4193271183.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


,movie title - year,rating,tags
0,Flaming Ears - 1992,6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,5.8,Six people - three couples - meet at random at...
3,Gulliver Returns - 2021,4.4,The legendary Gulliver returns to the Kingdom ...
5,Effie Gray - 2014,6.0,A look at the scandalous love triangle between...
6,Indigo - 2008,3.8,"A new generation of kids, called Indigo, who h..."


In [27]:
new_df.iloc[0]['tags']

'Flaming Ears is a pop sci-fi lesbian fantasy feature set in the year 2700 in the fictive burned-out city of Asche. It follows the tangled lives of three women - Volley, Nun and Spy. Fantasy, Sci-Fi'

In [28]:
# Converting to lower case
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

C:\Users\Laptop Lenovo\AppData\Local\Temp\ipykernel_14064\3444714728.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


In [29]:
new_df.head()

,movie title - year,rating,tags
0,Flaming Ears - 1992,6.0,flaming ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,5.8,six people - three couples - meet at random at...
3,Gulliver Returns - 2021,4.4,the legendary gulliver returns to the kingdom ...
5,Effie Gray - 2014,6.0,a look at the scandalous love triangle between...
6,Indigo - 2008,3.8,"a new generation of kids, called indigo, who h..."


In [30]:
new_df.iloc[0]['tags']

'flaming ears is a pop sci-fi lesbian fantasy feature set in the year 2700 in the fictive burned-out city of asche. it follows the tangled lives of three women - volley, nun and spy. fantasy, sci-fi'

In [31]:
#stemming
ps = PorterStemmer()

In [32]:
def stems(text):
    T = []
    
    for i in text.split():
        T.append(ps.stem(i))
    
    return " ".join(T)

In [33]:
new_df['tags'] = new_df['tags'].apply(stems)

C:\Users\Laptop Lenovo\AppData\Local\Temp\ipykernel_14064\3973021881.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stems)


In [34]:
new_df.head()

,movie title - year,rating,tags
0,Flaming Ears - 1992,6.0,flame ear is a pop sci-fi lesbian fantasi feat...
1,Jeg elsker dig - 1957,5.8,six peopl - three coupl - meet at random at a ...
3,Gulliver Returns - 2021,4.4,the legendari gulliv return to the kingdom of ...
5,Effie Gray - 2014,6.0,a look at the scandal love triangl between vic...
6,Indigo - 2008,3.8,"a new gener of kids, call indigo, who have mor..."


In [35]:
new_df.iloc[0]['tags']

'flame ear is a pop sci-fi lesbian fantasi featur set in the year 2700 in the fictiv burned-out citi of asche. it follow the tangl live of three women - volley, nun and spy. fantasy, sci-fi'

In [36]:
#check for 5000 samples for now
new_df = new_df.sample(5000, random_state=42).reset_index(drop=True)

In [37]:
#Vectorization 
#TFIDF Vector 
from sklearn.feature_extraction.text import TfidfVectorizer

v = TfidfVectorizer(max_features=3000, stop_words='english') 
transformed_output = v.fit_transform(new_df['tags']).toarray()

In [38]:
transformed_output

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(5000, 3000))

In [39]:
transformed_output.shape

(5000, 3000)

In [40]:
len(v.get_feature_names_out())

3000

In [41]:
from sklearn.metrics.pairwise import cosine_similarity

In [42]:
similarity = cosine_similarity(transformed_output)

In [43]:
similarity.shape

(5000, 5000)

In [44]:
#look for movies 
print(new_df['movie title - year'].sample(10).tolist())

['What a Hero! - 1992', 'Clank: Agent Recruit - 2015', 'Thor: God of Thunder - 2022', 'Independence Day: Resurgence - 2016', 'Invincible - 2006', "Hell on Devil's Island - 1957", 'Cherry - 1999', 'Hood Angels - 2003', 'The Castle of Sand - 1974', 'Aquí está Heraclio Bernal - 1958']


In [45]:
#location
new_df[new_df['movie title - year'] == 'Alien Presence - 2009'].index[0]

np.int64(2841)

In [46]:
#Recommend by title
def recommend(movie):
    movie = str(movie).lower()
    matches = new_df[new_df['movie title - year'].str.lower().str.contains(movie)]
    
    if matches.empty:
        print("No movie found")
        return
    
    idx = matches.index[0]
    distances = sorted(list(enumerate(similarity[idx])), reverse=True, key=lambda x: x[1])

    print(f"Best match is: {new_df.iloc[idx]['movie title - year']}")
    
    print(f"Recommendations are")
    # top 5 recommendations
    for i in distances[1:6]:
        print(new_df.iloc[i[0]]['movie title - year'])

In [47]:
recommend('Alien')

Best match is: STAR [Space Traveling Alien Reject] - 2017
Recommendations are
The Faces of My Gene - 2018
It's All Gone Pete Tong - 2004
What's Eating Todd? - 2016
Asmaan Se Ooncha - 1989
Behind the Random Denominator - 2017


In [48]:
#search by description and genres both 
def search_by_description(description):
    description_vector = v.transform([description.lower()])
    
    sim_scores = cosine_similarity(description_vector, transformed_output)
    
    distances = sorted(list(enumerate(sim_scores[0])), reverse=True, key=lambda x: x[1])
    
    print(f"\nTop matches for: '{description}'")
    print("-" * 30)
    
    # top 5 matches
    for i in distances[0:5]:
        print(f"{new_df.iloc[i[0]]['movie title - year']}")


In [49]:
# search by both description and genre 
search_by_description("horror")
search_by_description("oung gang member life around in ")


Top matches for: 'horror'
------------------------------
Jarring - 2009
The Evangelist - 2017
Puppet Master: Doktor Death - 2022
Phantasmagoria - 2014
Tales of the Creeping Death - 2022

Top matches for: 'oung gang member life around in '
------------------------------
Souk el selah - 1960
Jamesy Boy - 2014
Vorstadtkrokodile - 2009
Sin salida - 1971
3:15 the Moment of Truth - 1986
